In [1]:
import pandas as pd
import pickle
import numpy as np
import random
import networkx as nx
from sklearn.model_selection import KFold
import os
from tqdm.auto import tqdm
from collections import defaultdict
import torch
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv
from torch.utils.data import Dataset, DataLoader
from torch_geometric.loader import LinkNeighborLoader
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, average_precision_score, roc_auc_score
from torch.utils.data import Subset
from collections import deque
import time
import itertools

# Predict Bipartite Links using Pretrained model

## Model architecture

In [2]:
class ADRD_LinkPredictor(nn.Module):
    def __init__(self,
                 in_dims,
                 hidden_dim,
                 gat_heads,
                 fusion_heads,
                 dropout,
                 beta):
        super().__init__()
        self.beta = beta
        # A) Linear transforms
        self.type_linears = nn.ModuleDict({
            n: nn.Linear(in_dims[n], hidden_dim) for n in in_dims
        })
        self.shared_lin = nn.Linear(hidden_dim, hidden_dim)

        # All 6 relation keys
        self.rel_keys = [
            'drug-sim-drug', 'disease-sim-disease', 'protein-sim-protein',
            'drug-interacts-disease','disease-assoc-protein','drug-binds-protein'
        ]

        # B) 2-layer GATs
        self.gat1 = nn.ModuleDict({
            k: GATConv(hidden_dim, hidden_dim//gat_heads, heads=gat_heads, dropout = dropout,  concat=True)
            for k in self.rel_keys
        })
        self.gat2 = nn.ModuleDict({
            k: GATConv(hidden_dim, hidden_dim, heads=gat_heads, dropout = dropout, concat=False)
            for k in self.rel_keys
        })

        # C) Fusion self-attention (3→1)
        self.fusion_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=fusion_heads,
            batch_first=False,
            dropout=dropout
        )

        # D) Shared MLP
        self.link_mlp = nn.Sequential(
            nn.Linear(2*hidden_dim, hidden_dim),
            nn.ReLU(),  # no inplace
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, 1)
        )

    def compute_full_graph_embeddings(self, data):

        # A) Raw feature → shared space
        x = {}
        for ntype in data.node_types:
            h = F.relu(self.type_linears[ntype](data[ntype].x))
            x[ntype] = F.relu(self.shared_lin(h))

        # Prepare container for each node‐type’s list of [N_ntype×H] embeddings
        embs = {ntype: [] for ntype in data.node_types}

        # B) Run each of the 6 GATs
        for key in self.rel_keys:
            u_t, rel, v_t = key.split('-')
            c1, c2 = self.gat1[key], self.gat2[key]
            ei = data[u_t, rel, v_t].edge_index.to(x[u_t].device)

            if u_t == v_t:
                # similarity (homogeneous)
                h1 = F.elu(c1(x[u_t], ei))
                h2 = c2(h1, ei)                  # [N_u, H]
                embs[u_t].append(h2)

            else:
                # bipartite: ensure correct (src,dst) orientation
                N_u = x[u_t].size(0)
                N_v = x[v_t].size(0)
                max_src, max_dst = ei[0].max().item(), ei[1].max().item()
                if max_src >= N_u or max_dst >= N_v:
                    # flip rows if they got swapped at load time
                    ei = ei.flip(0).contiguous()

                # build the undirected view exactly once
                h_u, h_v = x[u_t], x[v_t]
                h_comb   = torch.cat([h_u, h_v], dim=0)  # [N_u+N_v, H]

                # forward edges map dst→[N_u..]
                ei_fwd = ei.clone()
                ei_fwd[1] += N_u

                # reverse edges map src→[N_u..]
                ei_rev = ei.flip(0).clone()
                ei_rev[0] += N_u

                ei_bid = torch.cat([ei_fwd, ei_rev], dim=1).to(h_comb.device)

                h1 = F.elu(c1(h_comb, ei_bid))
                h2 = c2(h1,    ei_bid)               # [N_u+N_v, H]

                # split back out
                embs[u_t].append(h2[:N_u])           # [N_u, H]
                embs[v_t].append(h2[N_u:])           # [N_v, H]

        # C) Fuse each node‐type’s 3 embeddings → 1
        H_fused = {}
        for ntype, lst in embs.items():
            # lst is 3 × [N_ntype, H]
            stacked = torch.stack(lst, dim=0)        # [3, N_ntype, H]
            fused, _ = self.fusion_attn(stacked, stacked, stacked)
            H_fused[ntype] = fused.mean(dim=0)       # [N_ntype, H]

        # —— NEW PART —— Combine into one giant embedding + record offsets
        H_drug = H_fused['drug']                    # [N_drug, H]
        H_dis  = H_fused['disease']                 # [N_disease, H]
        H_prot = H_fused['protein']                 # [N_protein, H]

        offsets = {
            'drug':    0,
            'disease': H_drug.size(0),
            'protein': H_drug.size(0) + H_dis.size(0)
        }

        H_all = torch.cat([H_drug, H_dis, H_prot], dim=0)  # [N_drug+N_disease+N_protein, H]

        return H_all, offsets


    # def forward(self, H_fused, u_types, v_types, u_idx, v_idx):
    #     # D) gather & MLP
    #     hu = torch.stack([H_fused[ut][i] for ut,i in zip(u_types, u_idx)], dim=0)
    #     hv = torch.stack([H_fused[vt][j] for vt,j in zip(v_types, v_idx)], dim=0)
    #     x  = torch.cat([hu, hv], dim=-1)
    #     return self.link_mlp(x).squeeze(-1)

    def forward(self, H_all, offsets, u_types, v_types, u_idx, v_idx):
        # Build per‐sample offsets
        off_u = torch.tensor([offsets[ut] for ut in u_types], dtype=torch.long, device=H_all.device)
        off_v = torch.tensor([offsets[vt] for vt in v_types], dtype=torch.long, device=H_all.device)

        # Compute global indices
        u_glob = u_idx + off_u
        v_glob = v_idx + off_v

        # Gather embeddings in one go
        hu = H_all[u_glob]   # [B, H]
        hv = H_all[v_glob]   # [B, H]

        x  = torch.cat([hu, hv], dim=-1)  # [B, 2H]
        return self.link_mlp(x).squeeze(-1)


    def compute_loss(self, logits, labels, rel_types):
        # E) weighted BCE
        sim = {'drug-sim-drug','disease-sim-disease','protein-sim-protein'}
        is_sim = torch.tensor([rt in sim for rt in rel_types], device=logits.device)
        is_bip = ~is_sim
        fn = F.binary_cross_entropy_with_logits
        l_sim = fn(logits[is_sim], labels[is_sim]) if is_sim.any() else 0.0
        l_bip = fn(logits[is_bip], labels[is_bip]) if is_bip.any() else 0.0
        return self.beta * l_bip + (1-self.beta) * l_sim

# ──────────────────────────────────────────────────────────────────────────────
# 4) Edge‐batch Dataset & DataLoader
# ──────────────────────────────────────────────────────────────────────────────
class FullGraphLinkDataset(Dataset):
    def __init__(self, folds, fold_idx, split = "train"):
        self.entries = []
        for rel_key, fl in folds.items():
            pos = fl[fold_idx][f"{split}_pos"].t().tolist()
            neg = fl[fold_idx][f"{split}_neg"].t().tolist()
            u_t, _, v_t = rel_key.split('-')
            for u,v in pos: self.entries.append((rel_key, u_t, v_t, u, v, 1.0))
            for u,v in neg: self.entries.append((rel_key, u_t, v_t, u, v, 0.0))

    def __len__(self): return len(self.entries)
    def __getitem__(self, i):
        rel_key, u_t, v_t, u, v, lbl = self.entries[i]
        return rel_key, u_t, v_t, torch.tensor(u), torch.tensor(v), torch.tensor(lbl)

## Node ID indexing

In [3]:
device = torch.device("cuda:2" if torch.cuda.device_count() > 1 else ("cuda" if torch.cuda.is_available() else "cpu"))

# ======================
# 1. Load node name lists & create mappings
# ======================
with open("drugs.pkl", "rb") as f:
    drugs = pickle.load(f)

with open("diseases.pkl", "rb") as f:
    diseases = pickle.load(f)

with open("proteins.pkl", "rb") as f:
    proteins = pickle.load(f)

num_drug = len(drugs)
num_disease = len(diseases)
num_protein = len(proteins)

offsets = {
    "drug": 0,
    "disease": num_drug,
    "protein": num_drug + num_disease
}

total_nodes = num_drug + num_disease + num_protein
print(f"Nodes: drugs={num_drug}, diseases={num_disease}, proteins={num_protein}, total={total_nodes}")

# Reverse mapping from global ID → original name
def global_id_to_name(node_id: int):
    if node_id < offsets["disease"]:
        return drugs[node_id]
    elif node_id < offsets["protein"]:
        return diseases[node_id - offsets["disease"]]
    else:
        return proteins[node_id - offsets["protein"]]

Nodes: drugs=2285, diseases=912, proteins=4042, total=7239


## Load model and Node Embeddings

In [4]:
# ======================
# 2. Load trained model + H_fused embeddings
# ======================
model = ADRD_LinkPredictor(
    in_dims      = {'drug':768, 'disease':768, 'protein':1280},
    hidden_dim   = 1024,
    gat_heads    = 8,
    fusion_heads = 8,
    beta         = 0.75,
    dropout      = 0.35
)

model.load_state_dict(torch.load("model_fold1.pth", map_location=device))
model.eval().to(device)

H_fused = torch.load("node_embeddings_fold1.pt", map_location=device)  # shape [total_nodes, emb_dim]
print("Loaded H_fused:", H_fused.shape)

Loaded H_fused: torch.Size([7239, 1024])


## Prepare all possible bipartite pairs

In [5]:
# ======================
# 3. Build global ID ranges for each node type
# ======================
drug_idx_global    = torch.arange(offsets["drug"], offsets["disease"])
disease_idx_global = torch.arange(offsets["disease"], offsets["protein"])
protein_idx_global = torch.arange(offsets["protein"], total_nodes)

# Cartesian products for all bipartite relations
drug_disease_pairs_global   = torch.cartesian_prod(drug_idx_global, disease_idx_global)
disease_protein_pairs_global = torch.cartesian_prod(disease_idx_global, protein_idx_global)
drug_protein_pairs_global    = torch.cartesian_prod(drug_idx_global, protein_idx_global)

print(f"Candidate pairs:")
print(f"  Drug–Disease = {drug_disease_pairs_global.shape[0]}")
print(f"  Disease–Protein = {disease_protein_pairs_global.shape[0]}")
print(f"  Drug–Protein = {drug_protein_pairs_global.shape[0]}")

Candidate pairs:
  Drug–Disease = 2083920
  Disease–Protein = 3686304
  Drug–Protein = 9235970


## Link Prediction using pretrained model

In [6]:
# ======================
# 4. Prediction function for global pairs
# ======================
@torch.no_grad()
def predict_links_for_pairs_unified(H_fused, model, pairs, batch_size=4096):
    preds = []
    for start in range(0, pairs.size(0), batch_size):
        end = min(start + batch_size, pairs.size(0))
        batch_pairs = pairs[start:end]

        hu = H_fused[batch_pairs[:,0]].to(device)
        hv = H_fused[batch_pairs[:,1]].to(device)

        x = torch.cat([hu, hv], dim=-1)   # [batch, 2*F]
        logits = model.link_mlp(x)
        probs = torch.sigmoid(logits).cpu()
        preds.append(probs)

        if (start // batch_size) % 100 == 0:
            print(f"  Predicted {end}/{pairs.size(0)} edges")

    return torch.cat(preds, dim=0)

In [ ]:
# ======================
# 5. Predict scores for each bipartite relation
# ======================
print("\nPredicting Drug–Disease edges...")
drug_disease_scores = predict_links_for_pairs_unified(H_fused, model, drug_disease_pairs_global)

print("\nPredicting Disease–Protein edges...")
disease_protein_scores = predict_links_for_pairs_unified(H_fused, model, disease_protein_pairs_global)

print("\nPredicting Drug–Protein edges...")
drug_protein_scores = predict_links_for_pairs_unified(H_fused, model, drug_protein_pairs_global)

# ======================
# 6. Threshold & collect predicted edges
# ======================
# Single threshold value
threshold = 0.5  

# Create masks ONCE
mask_dd = drug_disease_scores.squeeze(-1) > threshold
mask_dp = disease_protein_scores.squeeze(-1) > threshold
mask_dr = drug_protein_scores.squeeze(-1) > threshold

# Use same masks for edges
pred_drug_disease_edges   = drug_disease_pairs_global[mask_dd]
pred_disease_protein_edges = disease_protein_pairs_global[mask_dp]
pred_drug_protein_edges    = drug_protein_pairs_global[mask_dr]

predicted_bipartite_edges = torch.cat([
    pred_drug_disease_edges,
    pred_disease_protein_edges,
    pred_drug_protein_edges
], dim=0)

# Use SAME masks for scores
predicted_scores = torch.cat([
    drug_disease_scores.squeeze(-1)[mask_dd],
    disease_protein_scores.squeeze(-1)[mask_dp],
    drug_protein_scores.squeeze(-1)[mask_dr]
], dim=0)

print(f"Total predicted bipartite edges: {predicted_scores.size(0)}")

# torch.save(
#     {"edges": predicted_bipartite_edges, "scores": predicted_scores},
#     "predicted_bipartite_edges.pt"
# )

# Save Predicted Bipartite Edges

In [13]:
#Load lists of FDA-approved drugs, Neurological diseases, and assicated proteins
with open('drugs.pkl', 'rb') as f:
    drugs = pickle.load(f)

with open('diseases.pkl', 'rb') as f:
    diseases = pickle.load(f)

with open('proteins.pkl', 'rb') as f:
    proteins = pickle.load(f)


#Create mapping from name → integer index for each node type
drug2idx    = {name: i for i, name in enumerate(drugs)}
disease2idx = {name: i for i, name in enumerate(diseases)}
prot2idx    = {name: i for i, name in enumerate(proteins)}

In [39]:
#Load lists of FDA-approved drugs, Neurological diseases, and assicated proteins
with open('drugs.pkl', 'rb') as f:
    drugs = pickle.load(f)

with open('diseases.pkl', 'rb') as f:
    diseases = pickle.load(f)

with open('proteins.pkl', 'rb') as f:
    proteins = pickle.load(f)


# ─── 1) Counts & Offsets ───────────────────────────────────────────────────────
num_drug    = len(drugs)
num_disease = len(diseases)
num_protein = len(proteins)

offsets = {
    "drug":    0,
    "disease": num_drug,
    "protein": num_drug + num_disease
}

# ─── 2) Helper: map a global node‐ID back to its original name ─────────────────
def global_id_to_name(node_id: int):
    if node_id < offsets["disease"]:
        return drugs[node_id]
    elif node_id < offsets["protein"]:
        return diseases[node_id - offsets["disease"]]
    else:
        return proteins[node_id - offsets["protein"]]

# ─── 3) Build the global bipartite‐pair tensors ────────────────────────────────
drug_idx    = torch.arange(num_drug)
disease_idx = torch.arange(num_disease)
protein_idx = torch.arange(num_protein)

# local Cartesian products
local_dd = torch.cartesian_prod(drug_idx, disease_idx)   # [num_drug*num_disease, 2]
local_dp = torch.cartesian_prod(disease_idx, protein_idx)
local_dr = torch.cartesian_prod(drug_idx, protein_idx)

# convert to global IDs
global_dd = local_dd.clone()
global_dd[:, 0] += offsets["drug"]
global_dd[:, 1] += offsets["disease"]

global_dp = local_dp.clone()
global_dp[:, 0] += offsets["disease"]
global_dp[:, 1] += offsets["protein"]

global_dr = local_dr.clone()
global_dr[:, 0] += offsets["drug"]
global_dr[:, 1] += offsets["protein"]

# ─── 4) Assume you already have these tensors from predict_links_for_pairs_unified ─
#   drug_disease_scores : [num_drug*num_disease, 1]
#   disease_protein_scores: [num_disease*num_protein, 1]
#   drug_protein_scores  : [num_drug*num_protein, 1]

# ─── 5) Threshold & mask ───────────────────────────────────────────────────────
threshold = 0.5

mask_dd = drug_disease_scores.squeeze(-1)  > threshold  # [N_dd]
mask_dp = disease_protein_scores.squeeze(-1) > threshold
mask_dr = drug_protein_scores.squeeze(-1)  > threshold

# select only predicted edges
sel_dd_pairs  = global_dd[mask_dd]    # [M_dd, 2]
sel_dd_scores = drug_disease_scores.squeeze(-1)[mask_dd]

sel_dp_pairs  = global_dp[mask_dp]
sel_dp_scores = disease_protein_scores.squeeze(-1)[mask_dp]

sel_dr_pairs  = global_dr[mask_dr]
sel_dr_scores = drug_protein_scores.squeeze(-1)[mask_dr]

# ─── 6) Build & save DataFrames ────────────────────────────────────────────────

# Drug–Disease
df_dd = pd.DataFrame({
    'source': [global_id_to_name(int(u)) for u in sel_dd_pairs[:,0].tolist()],
    'target': [global_id_to_name(int(v)) for v in sel_dd_pairs[:,1].tolist()],
    'score' : sel_dd_scores.tolist()
})
df_dd.to_csv('predicted_drug_disease.csv', index=False)

# Disease–Protein
df_dp = pd.DataFrame({
    'source': [global_id_to_name(int(u)) for u in sel_dp_pairs[:,0].tolist()],
    'target': [global_id_to_name(int(v)) for v in sel_dp_pairs[:,1].tolist()],
    'score' : sel_dp_scores.tolist()
})
df_dp.to_csv('predicted_disease_protein.csv', index=False)

# Drug–Protein
df_dr = pd.DataFrame({
    'source': [global_id_to_name(int(u)) for u in sel_dr_pairs[:,0].tolist()],
    'target': [global_id_to_name(int(v)) for v in sel_dr_pairs[:,1].tolist()],
    'score' : sel_dr_scores.tolist()
})
df_dr.to_csv('predicted_drug_protein.csv', index=False)

print("Saved:")
print(f" • predicted_drug_disease.csv    ({len(df_dd)} edges)")
print(f" • predicted_disease_protein.csv ({len(df_dp)} edges)")
print(f" • predicted_drug_protein.csv    ({len(df_dr)} edges)")


Saved:
 • predicted_drug_disease.csv    (794837 edges)
 • predicted_disease_protein.csv (1641868 edges)
 • predicted_drug_protein.csv    (3036013 edges)


In [8]:
pred_drug_disease_edges.shape, pred_disease_protein_edges.shape, pred_drug_protein_edges.shape

(torch.Size([794837, 2]), torch.Size([1641868, 2]), torch.Size([3036013, 2]))

# Add back similarity links (Known)

In [9]:
device = torch.device("cuda:2" if torch.cuda.device_count() > 1 else ("cuda" if torch.cuda.is_available() else "cpu"))

# ======================
# 1. Load node name lists & create mappings
# ======================
with open("drugs.pkl", "rb") as f:
    drugs = pickle.load(f)

with open("diseases.pkl", "rb") as f:
    diseases = pickle.load(f)

with open("proteins.pkl", "rb") as f:
    proteins = pickle.load(f)

num_drug = len(drugs)
num_disease = len(diseases)
num_protein = len(proteins)

offsets = {
    "drug": 0,
    "disease": num_drug,
    "protein": num_drug + num_disease
}

total_nodes = num_drug + num_disease + num_protein
print(f"Nodes: drugs={num_drug}, diseases={num_disease}, proteins={num_protein}, total={total_nodes}")

# Reverse mapping from global ID → original name
def global_id_to_name(node_id: int):
    if node_id < offsets["disease"]:
        return drugs[node_id]
    elif node_id < offsets["protein"]:
        return diseases[node_id - offsets["disease"]]
    else:
        return proteins[node_id - offsets["protein"]]

Nodes: drugs=2285, diseases=912, proteins=4042, total=7239


## Load Predicted Bipartite Links

In [10]:
# ────────────────────────────────────────────────────────────────
# 2) Load  predicted bipartite edges & scores
# ────────────────────────────────────────────────────────────────
predicted_data  = torch.load("predicted_bipartite_edges.pt")
predicted_scores = predicted_data["scores"]
predicted_edges  = predicted_data["edges"]

In [11]:
# ────────────────────────────────────────────────────────────────
# 3) Helper to load & normalize one similarity file
# ────────────────────────────────────────────────────────────────
def load_and_norm_sim(csv_path, idx_map, src_offset, dst_offset):
    df = pd.read_csv(csv_path)
    # Rename columns to standard names regardless of original
    df.columns = ["n1", "n2", "score"]
    # map names → local idx, then to global
    df["u"] = df["n1"].map(idx_map) + src_offset
    df["v"] = df["n2"].map(idx_map) + dst_offset
    # drop any unmapped
    df = df.dropna(subset=["u","v"])
    df["u"] = df["u"].astype(int)
    df["v"] = df["v"].astype(int)
    # min-max normalize score → [0,1]
    smin, smax = df["score"].min(), df["score"].max()
    df["w"] = (df["score"] - smin) / (smax - smin)
    # return edge‐tensor and weight‐tensor
    edges  = torch.tensor(df[["u","v"]].values.T, dtype=torch.long)
    weights= torch.tensor(df["w"].values, dtype=torch.float)
    return edges, weights

In [12]:
# ────────────────────────────────────────────────────────────────
# 4) Load all three similarity relations
# ────────────────────────────────────────────────────────────────
# you must have created these earlier:
drug2idx, disease2idx, prot2idx = (
    {n:i for i,n in enumerate(drugs)},
    {n:i for i,n in enumerate(diseases)},
    {n:i for i,n in enumerate(proteins)},
)

dd_edges, dd_w = load_and_norm_sim(
    "drug_sim.csv",
    drug2idx, offsets["drug"], offsets["drug"]
)
ds_edges, ds_w = load_and_norm_sim(
    "disease_sim.csv",
    disease2idx, offsets["disease"], offsets["disease"]
)
pp_edges, pp_w = load_and_norm_sim(
    "protein_sim.csv",
    prot2idx, offsets["protein"], offsets["protein"]
)

In [13]:
dd_edges.shape, ds_edges.shape, pp_edges.shape

(torch.Size([2, 3572100]), torch.Size([2, 367653]), torch.Size([2, 561890]))

In [14]:
# ────────────────────────────────────────────────────────────────
# 5) Merge bipartite + similarity edges & weights
# ────────────────────────────────────────────────────────────────
all_edges  = torch.cat([
    predicted_edges.t(),
    dd_edges, ds_edges, pp_edges
], dim=1)  # -> [2, M_total]

all_weights = torch.cat([
    predicted_scores,
    dd_w, ds_w, pp_w
], dim=0)  # -> [M_total]


# Build NetworkX object

In [17]:
# ────────────────────────────────────────────────────────────────
# 6) Build a NetworkX graph for (biased) PageRank
# ────────────────────────────────────────────────────────────────
G = nx.Graph()
G.add_nodes_from(range(total_nodes))
u_idx, v_idx = all_edges
for u,v,w in zip(u_idx.tolist(), v_idx.tolist(), all_weights.tolist()):
    G.add_edge(u, v, weight=w)

In [16]:
total_nodes

7239

In [31]:
print("Graph built: ", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

Graph built:  7239 nodes, 7908311 edges


# PageRank on ADRD diseases

In [32]:
disease_name_dict = {
                'D000544' : 'Alzheimer Disease',
                'D015140' : 'Vascular Dementia',
                'D010300' : 'Parkinson Disease Dementia',
                'D057180' : 'Frontotemporal Dementia',
                'D006816' : 'Huntington Disease',
                'D006850' : 'Normal Pressure Hydrocephalus',
                'D020961' : 'Lewy Body Disease'
                    }

In [65]:
disease_name_dict = {
                'D000544' : 'Alzheimer Disease',
                'D015140' : 'Vascular Dementia',
                    }

In [ ]:
# Node Id Offsets
num_drug    = offsets["disease"]
num_disease = offsets["protein"] - offsets["disease"]

for dcode, dname in disease_name_dict.items():
    print(dcode)
    if dcode not in disease2idx:
        print(f" {dcode} not found, skipping")
        continue

    # global ID of this disease
    local_idx = disease2idx[dcode]
    global_id = offsets["disease"] + local_idx

    # --- 2-hop neighborhood (will include proteins automatically) ---
    # this returns a dict {node: distance}, so .keys() is all nodes ≤2 hops away
    nodes_2hop = nx.single_source_shortest_path_length(G, global_id, cutoff=2).keys()
    G_sub = G.subgraph(nodes_2hop).copy()

    # --- personalized PageRank on that induced subgraph ---
    pers = {n: 0.0 for n in G_sub.nodes()}
    pers[global_id] = 1.0

    pr = nx.pagerank(G_sub, alpha=0.85, personalization=pers, weight="weight")

    # --- extract only drugs for  ranking (but proteins WERE used in the subgraph) ---
    drug_pr = {
        global_id_to_name(n): score
        for n, score in pr.items()
        if n < num_drug
    }

    # top‐10 drugs
    top_drugs = sorted(drug_pr.items(), key=lambda x: -x[1])[:10]
    print(f"\n Top 10 drugs for {dname} ({dcode}) in its 2-hop subgraph:")
    for drug_name, score in top_drugs:
        print(f"   • {drug_name:<30}  PR = {score:.4f}")


In [42]:
import networkx as nx
from collections import deque
import math

def multi_source_bfs_distances(G, sources):
    dist = {}
    dq   = deque()
    for s in sources:
        dist[s] = 0
        dq.append(s)
    while dq:
        u = dq.popleft()
        for v in G.neighbors(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                dq.append(v)
    return dist

# assume G, drugs, disease2idx, proteins, offsets, global_id_to_name, disease_name_dict

# build pure protein‐protein subgraph
prot_start = offsets["protein"]
prot_end   = prot_start + len(proteins)
H_prot     = G.subgraph(range(prot_start, prot_end)).copy()

for dcode, dname in disease_name_dict.items():
    if dcode not in disease2idx:
        print(f" {dcode} not in idx, skipping")
        continue

    g_d = offsets["disease"] + disease2idx[dcode]
    # disease→protein neighbors in G
    disease_proteins = [nbr for nbr in G.neighbors(g_d)
                        if prot_start <= nbr < prot_end]
    if not disease_proteins:
        print(f" No protein neighbors for {dcode}, skipping")
        continue

    # BFS distances from all disease proteins
    dist = multi_source_bfs_distances(H_prot, disease_proteins)

    proximities = []
    for i, drug_name in enumerate(drugs):
        g_u = offsets["drug"] + i
        # drug→protein neighbors
        drug_proteins = [nbr for nbr in G.neighbors(g_u)
                         if prot_start <= nbr < prot_end]
        if not drug_proteins:
            dc, md = math.inf, math.inf
        else:
            ds = [dist.get(p, math.inf) for p in drug_proteins]
            dc     = min(ds)
            md     = sum(ds) / len(ds)
        proximities.append((drug_name, dc, md))

    # rank by closest proximity
    proximities.sort(key=lambda x: (x[1], x[2]))
    print(f"\n🔍 Top 10 drugs by closest proximity to {dname} ({dcode}):")
    for name, dc, md in proximities[:10]:
        print(f"  • {name:<30}  d_c={dc}, mean_d={md:.2f}")



🔍 Top 10 drugs by closest proximity to Alzheimer Disease (D000544):
  • DB13757                         d_c=0, mean_d=0.00
  • DB06152                         d_c=0, mean_d=0.00
  • DB00771                         d_c=0, mean_d=0.00
  • DB12924                         d_c=0, mean_d=0.00
  • DB04825                         d_c=0, mean_d=0.00
  • DB00405                         d_c=0, mean_d=0.00
  • DB00719                         d_c=0, mean_d=0.00
  • DB00902                         d_c=0, mean_d=0.00
  • DB01237                         d_c=0, mean_d=0.00
  • DB01246                         d_c=0, mean_d=0.00

🔍 Top 10 drugs by closest proximity to Vascular Dementia (D015140):
  • DB13757                         d_c=0, mean_d=0.00
  • DB06152                         d_c=0, mean_d=0.00
  • DB00771                         d_c=0, mean_d=0.00
  • DB12924                         d_c=0, mean_d=0.00
  • DB04825                         d_c=0, mean_d=0.00
  • DB00283                         d

# Drug Combination Ranking

In [68]:
# ─── 0) BFS helper ────────────────────────────────────────────────────────────
def multi_source_bfs_distances(G, sources):
    dist = {}
    dq   = deque(sources)
    for s in sources:
        dist[s] = 0
    while dq:
        u = dq.popleft()
        for v in G.neighbors(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                dq.append(v)
    return dist

# ─── 1) Build protein–protein similarity subgraph ────────────────────────────
prot_start = offsets["protein"]
prot_end   = prot_start + len(proteins)
H_prot     = G.subgraph(range(prot_start, prot_end)).copy()

# ─── 2) Ingest predicted drug→protein edges → drug_targets ───────────────────
# pred_drug_protein_edges : LongTensor[M,2] of GLOBAL IDs
drug_targets = { name: set() for name in drugs }
for u_glob, v_glob in pred_drug_protein_edges.cpu().tolist():
    # identify drug vs protein
    if u_glob < offsets["disease"]:
        drug_glob, prot_glob = u_glob, v_glob
    else:
        drug_glob, prot_glob = v_glob, u_glob
    drug_local = drug_glob - offsets["drug"]     # 0..num_drugs-1
    prot_local = prot_glob - offsets["protein"]  # 0..num_proteins-1
    drug_name  = drugs[drug_local]
    drug_targets[drug_name].add(prot_local)

# ─── 3) Ingest predicted drug→disease edges → drug_diseases ────────────────
# pred_drug_disease_edges : LongTensor[M,2]
drug_diseases = { name: set() for name in drugs }
for u_glob, v_glob in pred_drug_disease_edges.cpu().tolist():
    # identify drug vs disease
    if u_glob < offsets["disease"]:
        drug_glob, dis_glob = u_glob, v_glob
    else:
        drug_glob, dis_glob = v_glob, u_glob
    drug_local = drug_glob - offsets["drug"]
    dis_local  = dis_glob  - offsets["disease"]
    drug_name  = drugs[drug_local]
    disease_code = diseases[dis_local]
    drug_diseases[drug_name].add(disease_code)

# ─── 4) Load disease–disease similarity matrix for DSM ────────────────────────
# ─── 4) Load disease–disease similarity triples and build DSIM matrix ───────
# Your CSV has no header, e.g. rows like: D000544,D015140,0.87
triples = pd.read_csv("disease_sim.csv")

num_dis = len(diseases)
DSIM = np.zeros((num_dis, num_dis), dtype=float)

# fill in both [i,j] and [j,i]
for _, row in triples.iterrows():
    a, b = row["Disease1"], row["Disease2"]
    s     = float(row["Sim_score"])
    if a in disease2idx and b in disease2idx:
        i = disease2idx[a]
        j = disease2idx[b]
        DSIM[i, j] = s
        DSIM[j, i] = s

# normalize all nonzero scores into [0,1]
nonz = DSIM[DSIM > 0]
if len(nonz):
    mn, mx = nonz.min(), nonz.max()
    DSIM[DSIM > 0] = (DSIM[DSIM > 0] - mn) / (mx - mn)

# ─── 5) Metric definitions ──────────────────────────────────────────────────
def jaccard_overlap(A, B):
    if not A and not B:
        return 0.0
    inter = len(A & B)
    union = len(A | B)
    return inter / union

def internal_mean_distance(G, nodes_glob):
    if len(nodes_glob) < 2:
        return 0.0
    total = 0
    count = 0
    for u, v in itertools.combinations(nodes_glob, 2):
        try:
            total += nx.shortest_path_length(G, u, v)
        except nx.NetworkXNoPath:
            total += math.inf
        count += 1
    return total / count

def mean_distance(G, A_glob, B_glob):
    if not A_glob or not B_glob:
        return math.inf
    total = 0
    count = 0
    for a in A_glob:
        dist_map = nx.single_source_shortest_path_length(G, a)
        for b in B_glob:
            total += dist_map.get(b, math.inf)
            count += 1
    return total / count

# ─── 6) Compute combo rankings per disease ──────────────────────────────────
α, β, γ = 1.0, 1.0, 1.0  # weights for OM, CM, and DSM

for dcode, dname in disease_name_dict.items():
    g_d = offsets["disease"] + disease2idx[dcode]
    disease_prots = [nbr for nbr in G.neighbors(g_d)
                     if prot_start <= nbr < prot_end]
    if not disease_prots:
        print(f" No protein neighbors for {dcode}, skipping")
        continue

    # pre‐BFS on H_prot (optional for CM caching)
    _dist_module = multi_source_bfs_distances(H_prot, disease_prots)

    results = []
    for drugA, drugB in itertools.combinations(drugs, 2):
        # A) Overlap Metric
        TA = drug_targets[drugA]
        TB = drug_targets[drugB]
        OM = jaccard_overlap(TA, TB)

        # convert targets to global IDs for PPI distances
        TA_glob = {prot_start + p for p in TA}
        TB_glob = {prot_start + p for p in TB}

        # B) Complementary Metric
        dAB = mean_distance(H_prot, TA_glob, TB_glob)
        dAA = internal_mean_distance(H_prot, TA_glob)
        dBB = internal_mean_distance(H_prot, TB_glob)
        CM  = dAB - 0.5*(dAA + dBB)

        # C) Disease‐Similarity Metric
        Da = [disease2idx[x] for x in drug_diseases[drugA] if x in disease2idx]
        Db = [disease2idx[x] for x in drug_diseases[drugB] if x in disease2idx]
        if Da and Db:
            vals = [DSIM[i,j] for i in Da for j in Db]
            DSM = float(np.mean(vals))
        else:
            DSM = 0.0

        # D) Unified combination score
        # combo_score = α*(1-OM) + β*CM + γ*DSM

        combo_score = ((β*CM) / (α*OM)) *  (γ*DSM)

        results.append({
            "drugA": drugA, "drugB": drugB,
            "OM": OM, "CM": CM, "DSM": DSM,
            "score": combo_score
        })

    # sort by descending unified score
    ranking = sorted(results, key=lambda x: -x["score"])

    print(f"\n Top 10 combos for {dname} ({dcode}):")
    for rec in ranking[:10]:
        print(f"  • {rec['drugA']} + {rec['drugB']:<20}"
              f" OM={rec['OM']:.2f}, CM={rec['CM']:.2f}, DSM={rec['DSM']:.2f},"
              f" score={rec['score']:.3f}")

KeyboardInterrupt: 

In [ ]:
metrics_summary_mean = {
    ds: {
        metric: round(np.mean(scores), 4)
        for metric, scores in metrics.items()
    }
    for ds, metrics in metrics_summary.items()
}
metrics_summary_mean

{'C': {'F1': 0.9543, 'AUC': 0.9858, 'AUPR': 0.9893},
 'F': {'F1': 0.9578, 'AUC': 0.9862, 'AUPR': 0.9898},
 'Y': {'F1': 0.9219, 'AUC': 0.9741, 'AUPR': 0.9776},
 'LRSSL': {'F1': 0.9863, 'AUC': 0.9962, 'AUPR': 0.9973},
 'LAGCN': {'F1': 0.8315, 'AUC': 0.9057, 'AUPR': 0.8887}}